# Tutorial: IBM CLEAR with Eval Hub (local + deployed)

## Quick start (run order)

| Goal | Run |
|------|-----|
| **Local CLEAR only** | **Step 1** → **Step 2** (traces) → **Step 3** (job + `main.py`) → **Step 4**. Stop before **Part B**. You need **`examples/.env`** with **`MODEL_URL`** / **`MODEL_NAME`** (Step 3 loads it if **`python-dotenv`** is installed). |
| **Eval Hub (`curl` or SDK)** | Complete Part A if you want; then open **Part B**. For the **SDK**, run the **`%pip`** cell, **restart the kernel**, then **Step 1** again → load **`.env`** → continue in order. |

**Where outputs go:** **`examples/output/local/`** — Part A (`clear_results.json`, log, job spec). **`examples/output/evalhub/`** — Part B SDK snapshots (metrics JSON, job payloads); not a copy of **`clear_results.json`** from the cluster unless you add MLflow/OCI on the job.

---

In this notebook we go through **running CLEAR on JSON agent traces on your machine** (Part A), then **what a deployed Eval Hub job looks like** — **`curl`** first, then an **optional Python SDK** path (Part B). 

**What is CLEAR?** **CLEAR** (**Comprehensive LLM Error Analysis and Reporting**, IBM) is a pipeline that ingests **agent traces** (often MLflow-style JSON), uses an **LLM as a judge** to score and cluster interactions, and surfaces **recurring failure patterns**. Its main artifact is **`clear_results.json`** — a structured report of **what went wrong and how often**, not just a raw dump of traces.

**What is Eval Hub?** **Eval Hub** is the **service** that **schedules** evaluation work on a cluster: it takes a **job spec** (model URL, benchmark, optional **object storage** for traces), runs adapters like **ibm-clear**, and returns **metrics** when the job completes.

**Flow**

1. **Step 1 — Paths** — Sets **`input-traces/`**, **`output/local/`** (Part A), and **`output/evalhub/`** (Part B SDK) whether you start Jupyter from **`adapters/clear`** or **`adapters/clear/examples`**.
2. **Part A (Steps 2–4)** — Build a **local job** from the **`meta/job.json`** template, run **`main.py`** via **`subprocess`** (same as the CLI), and tune **`USE_MLFLOW`** / **`INFERENCE_BACKEND`**.
3. **Part B** — **`curl`** against Eval Hub (shape of a deployed job), then **optional SDK** cells.

**Config:** Copy **`examples/env.example`** to **`examples/.env`**. Set **`MODEL_URL`**, **`MODEL_NAME`**, and (for Part B) **`EVALHUB_*`**, **`NAMESPACE`**, **`S3_*`** as needed. **`env.example`** is checked in; **`.env`** is git-ignored.

**Local-only path:** If you only want Part A, you can skip **Prerequisites** until you try Part B; you still want **`.env`** for **Part A** model URLs.

## Prerequisites: Eval Hub credentials (`.env`) — Part B

**Skip this section** if you are only doing **Part A** (local CLEAR). You still use **`examples/.env`** for **`MODEL_*`** in Step 3.

Use this when you run **Part B** (Eval Hub API or SDK). Put values in **`examples/.env`** — same file as **Part A** — so nothing sensitive sits in notebook cells.

**Template:** **`examples/env.example`** — copy to **`.env`**. Fill **`EVALHUB_*`**, **`NAMESPACE`**, **`MODEL_URL`**, **`MODEL_NAME`**, **`S3_*`** as you need.

**Getting a token and URL (OpenShift-style example — change namespace/route to yours):** from a shell, then merge lines into **`.env`** (by hand or with a here-doc):

```bash
export EVALHUB_TOKEN=$(oc whoami -t)
export EVALHUB_URL=https://$(oc get route evalhub -n "${NAMESPACE}" -o jsonpath='{.spec.host}')
export EVALHUB_TENANT="your-tenant"
# Set NAMESPACE in .env first; paste EVALHUB_* and MODEL_* into examples/.env
```

Eval Hub multi-tenancy: https://eval-hub.github.io/development/multi-tenancy

**OpenShift authentication:**

1. **ServiceAccount token** (`auth_token_path`): e.g. `/var/run/secrets/kubernetes.io/serviceaccount/token` in-cluster.
2. **User token** (`auth_token`): `oc whoami -t` for local dev.
3. **Environment variable**: production / CI secrets.

Kernel packages for Part B are installed later via the **`%pip`** cell (includes `typing_extensions` + `eval-hub-sdk`).

## Step 1 — Resolve paths (run this first)

Jupyter’s current working directory is often either **`adapters/clear`** or **`adapters/clear/examples`**. This step sets **`ADAPTER_ROOT`**, **`EXAMPLES_DIR`**, **`INPUT_TRACES`**, **`OUTPUT_DIR_LOCAL`** (**`examples/output/local/`** — Part A), and **`OUTPUT_DIR_EVALHUB`** (**`examples/output/evalhub/`** — Part B SDK). **Step 3** writes **`local-job-spec.json`**, the log, and (after a good run) **`clear_results.json`** under **`output/local/`**.

Part A sets **`results_dir`** = **`examples/output`** and **`run_name`** = **`local`** so CLEAR writes under **`output/local/`**. Part B saves API snapshots under **`output/evalhub/`**. **`curl`** from a terminal does **not** write those files — only this notebook’s SDK cells do.

In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import time

HERE = Path.cwd().resolve()
if HERE.name == "examples" and (HERE.parent / "main.py").is_file():
    ADAPTER_ROOT = HERE.parent
    EXAMPLES_DIR = HERE
elif (HERE / "main.py").is_file():
    ADAPTER_ROOT = HERE
    EXAMPLES_DIR = HERE / "examples"
else:
    raise RuntimeError(
        "Use working directory adapters/clear or adapters/clear/examples (folder containing main.py)."
    )

INPUT_TRACES = EXAMPLES_DIR / "input-traces"
_out_root = EXAMPLES_DIR / "output"
OUTPUT_DIR_LOCAL = _out_root / "local"   # Part A — main.py on this machine
OUTPUT_DIR_EVALHUB = _out_root / "evalhub"  # Part B — SDK snapshots from deployed Eval Hub
OUTPUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_EVALHUB.mkdir(parents=True, exist_ok=True)

# Part A runs main.py in a subprocess — use adapter .venv if it exists (CLEAR lives there).
_venv_py = ADAPTER_ROOT / ".venv" / "bin" / "python"
MAIN_PYTHON = str(_venv_py) if _venv_py.is_file() else sys.executable

print("Adapter root:", ADAPTER_ROOT)
print("Examples dir:", EXAMPLES_DIR)
print("Input traces:", INPUT_TRACES)
print("Part A output (local CLEAR):", OUTPUT_DIR_LOCAL)
print("Part B output (Eval Hub SDK files):", OUTPUT_DIR_EVALHUB)
print("Python for main.py subprocess:", MAIN_PYTHON)
if not _venv_py.is_file():
    print(
        "No .venv at adapter root — main.py uses the Jupyter kernel Python.\n"
        "Fix: cd adapters/clear && python3 -m venv .venv && .venv/bin/pip install -r requirements.txt\n"
        "Or: register that venv as your Jupyter kernel and select it here."
    )

### Before Part A: venv + how `main.py` runs

CLEAR and its dependencies are installed in the adapter’s **virtualenv** under **`adapters/clear/.venv`**, not automatically in the Jupyter kernel. **Part A** runs **`main.py`** as a **separate process** using that venv’s Python when it exists — same as typing **`python main.py`** in a terminal. That way you are debugging **one** code path (what ops run in local/k8s mode), and you get a single **log file** from the adapter instead of mixing CLEAR into notebook state.

**One-time (terminal):** from **`adapters/clear`** (where **`main.py`** and **`requirements.txt`** live):

```bash
python3 -m venv .venv && .venv/bin/pip install -U pip && .venv/bin/pip install -r requirements.txt
```

**Then in Jupyter:** restart the kernel, re-run **Step 1**, and check **Python for main.py subprocess** — it should show **`…/adapters/clear/.venv/bin/python`**.

**Why not paste `main.py` into a cell?** Keeps behavior aligned with **CLI / cluster** runs. **Part B** is different: the Eval Hub **SDK** runs **inside this kernel**, so later we **`%pip`** install it here — that is intentional.

---

# Part A — Local run (`main.py`)

**What we are actually doing:** **`meta/job.json`** is the adapter’s **reference job** — same fields Eval Hub would send on the cluster. We **load it as a template** and then **only swap what a laptop needs**: **`data_dir`** → **`input-traces/`**; **`results_dir`** / **`run_name`** → **`examples/output/local/`** (so **`clear_results.json`** lands next to the local job spec); **`test_data_ref`** is **removed** because that is for **object storage on the cluster**, not files on your disk.

**Toggles at the top of the run cell**

- **`USE_MLFLOW = False`** — typical local run without an experiment. Set **`True`** only if **`MLFLOW_TRACKING_URI`** and an experiment are set up.
- **`INFERENCE_BACKEND = "endpoint"`** — CLEAR uses your **`model.url`** (OpenAI-compatible **`/v1`**). You usually **skip `OPENAI_API_KEY`**. Use **`"litellm"`** if you want LiteLLM wiring and usually need the key in the environment.

**Steps:** **Step 2** — list traces → **`.env`** (**`env.example`** → **`.env`**, **`MODEL_URL`** / **`MODEL_NAME`**) → **Step 3** — write **`local-job-spec.json`** and run **`main.py`** → **Step 4** — confirm **`clear_results.json`**.

**`Connection refused`** on **`callback_url`** in the log is normal locally — nothing on your machine is listening for Eval Hub callbacks.

In [ ]:
# Step 2 — traces must exist under input-traces/ (top-level *.json only; use rglob if nested)
traces = sorted(INPUT_TRACES.glob("*.json"))
if not traces:
    raise FileNotFoundError(
        f"Add at least one *.json trace file under {INPUT_TRACES}."
    )
print(f"✓ Found {len(traces)} trace file(s):", [p.name for p in traces])

### `examples/.env` — model + cluster settings

**Template:** **`examples/env.example`** → **`examples/.env`**. Use **`MODEL_URL`** and **`MODEL_NAME`** only (one name for both **`model.name`** and **`eval_model_name`** in the job JSON). See **`env.example`** for Eval Hub, **`NAMESPACE`**, S3, and optional **`JOB_NAME`**. Do **not** commit **`.env`**.

**Step 3** loads **`.env`** when **`python-dotenv`** is installed (`pip install python-dotenv`, or the Part B **`%pip`** cell first). Otherwise **`export`** the same variables in your shell before Jupyter.

| Variable | Purpose |
|----------|---------|
| **`MODEL_URL`** | OpenAI-compatible API base (e.g. `http://127.0.0.1:8000/v1`). |
| **`MODEL_NAME`** | Same string for **`model.name`** and **`parameters.eval_model_name`**. |
| **`OPENAI_API_KEY`** | Only with **`litellm`**. With **`endpoint`** + **`model.url`**, omit (Part A and typical Part B).

**Part B** uses the same **`MODEL_*`** plus **`EVALHUB_*`**, **`NAMESPACE`**, **`S3_*`** — see **`env.example`** and **Prerequisites**.

In [ ]:
# Step 3 — job spec from meta/job.json, then run main.py
try:
    from dotenv import load_dotenv

    load_dotenv(EXAMPLES_DIR / ".env")
except ImportError:
    pass

meta_job = ADAPTER_ROOT / "meta" / "job.json"
job = json.loads(meta_job.read_text(encoding="utf-8"))

# --- Tune locally (same shape as meta/job.json) ---
USE_MLFLOW = False  # True only if MLFLOW_TRACKING_URI + experiment are set up
INFERENCE_BACKEND = "endpoint"  # "endpoint" → model.url, often no OPENAI_API_KEY; "litellm" → usually need the key

job["id"] = "clear-notebook-local-001"
job["parameters"] = dict(job["parameters"])
job["parameters"]["data_dir"] = str(INPUT_TRACES.resolve())
job["parameters"]["results_dir"] = str((EXAMPLES_DIR / "output").resolve())
job["parameters"]["run_name"] = "local"
job["parameters"]["inference_backend"] = INFERENCE_BACKEND
job.pop("test_data_ref", None)

if not USE_MLFLOW:
    job["experiment_name"] = None
    job["parameters"].pop("mlflow_experiment_name", None)

if INFERENCE_BACKEND == "endpoint":
    job["parameters"].pop("inference_url", None)
    if job.get("model", {}).get("auth"):
        job["model"] = dict(job["model"])
        job["model"].pop("auth", None)

_murl = os.getenv("MODEL_URL")
_mname = os.getenv("MODEL_NAME")
if _murl or _mname:
    job["model"] = dict(job.get("model") or {})
    if _murl:
        job["model"]["url"] = _murl.strip()
    if _mname:
        _mn = _mname.strip()
        job["model"]["name"] = _mn
        job["parameters"]["eval_model_name"] = _mn
    print("Applied from .env:", {k: v for k, v in [("MODEL_URL", _murl), ("MODEL_NAME", _mname)] if v})

job_path = OUTPUT_DIR_LOCAL / "local-job-spec.json"
job_path.write_text(json.dumps(job, indent=2), encoding="utf-8")

env = os.environ.copy()
env["EVALHUB_MODE"] = "local"
env["EVALHUB_JOB_SPEC_PATH"] = str(job_path.resolve())

log_path = OUTPUT_DIR_LOCAL / "local-main-py.log"
with log_path.open("w", encoding="utf-8") as logf:
    proc = subprocess.run(
        [MAIN_PYTHON, str(ADAPTER_ROOT / "main.py")],
        cwd=str(ADAPTER_ROOT),
        env=env,
        stdout=logf,
        stderr=subprocess.STDOUT,
        text=True,
    )

print(f"✓ Wrote job spec: {job_path}")
print(f"✓ Wrote log: {log_path}")
print(f"✓ Exit code: {proc.returncode} (see log if non-zero)")
if proc.returncode != 0:
    print("WARNING: main.py failed — open local-main-py.log for details.")

In [ ]:
# Step 4 — confirm CLEAR output
final_json = OUTPUT_DIR_LOCAL / "clear_results.json"
if final_json.is_file():
    print("CLEAR wrote:", final_json)
else:
    print("No clear_results.json yet — read local-main-py.log for errors.")

---

# Part B — Deployed Eval Hub

You need a **running Eval Hub** (HTTPS), a **Bearer** token, **`X-Tenant`**, and the **`ibm-clear`** provider registered. **Do not paste real cluster URLs into the notebook** — keep hosts, model URLs, and bucket names in **`examples/.env`** (start from **`env.example`**).

## Submit with `curl` (typical)

**Flow:** load **`EVALHUB_*`**, **`MODEL_URL`**, **`MODEL_NAME`**, **`S3_*`**, optional **`EXPERIMENT_NAME`**, **`export`**, then run **`curl`**. **`S3_PREFIX`** is the bucket prefix for your **`*.json`** keys. Staging often yields flat **`*.json`** under **`/test_data`**. **`inference_backend`** is **`endpoint`** (uses **`model.url`**, no pod **`OPENAI_API_KEY`**). Add **`parameters.data_dir`** only if needed.

```bash
export TOKEN="${EVALHUB_TOKEN:-$(oc whoami -t)}"
# source .env: EVALHUB_URL, EVALHUB_TENANT, MODEL_URL, MODEL_NAME, S3_*, EXPERIMENT_NAME

curl -s -k -X POST \\
  "${EVALHUB_URL}/api/v1/evaluations/jobs" \\
  -H "Authorization: Bearer ${TOKEN}" \\
  -H "X-Tenant: ${EVALHUB_TENANT}" \\
  -H "Content-Type: application/json" \\
  -d @- <<EOF
{
  "name": "clear-agentic-example",
  "model": {
    "url": "${MODEL_URL}",
    "name": "${MODEL_NAME}"
  },
  "experiment": {
    "name": "${EXPERIMENT_NAME:-clear-agentic-mlflow-exp}"
  },
  "benchmarks": [
    {
      "id": "agentic-evaluation",
      "provider_id": "ibm-clear",
      "parameters": {
        "eval_model_name": "${MODEL_NAME}",
        "provider": "openai",
        "agent_framework": "langgraph",
        "observability_framework": "mlflow",
        "inference_backend": "endpoint",
        "agent_mode": true
      },
      "test_data_ref": {
        "s3": {
          "bucket": "${S3_BUCKET}",
          "key": "${S3_PREFIX}",
          "secret_ref": "${S3_SECRET_REF}"
        }
      }
    }
  ]
}
EOF
```

**Note:** Use **`"litellm"`** here only if your platform requires it; then set **`OPENAI_API_KEY`** on the worker pod.

## Traces from S3 (deployment — same idea as `curl`)

**In this deployed pattern, input traces come from object storage**, not from a path on your laptop. You put **`test_data_ref.s3`** on the job (bucket, prefix **`key`** or **`path`**, and **`secret_ref`** for how the cluster reads the bucket). **Eval Hub / the operator** then **downloads** those objects and **lays them down as files** in the job pod (often under **`/test_data`** or **`/data`**). The CLEAR adapter in the pod **only sees a directory of `*.json`** — same as Part A, but the files arrived via staging, not via your notebook. **The notebook does not talk to S3 directly** for that job; it only shows the JSON shape (or **`curl`**) you send to Eval Hub.

**Map names to `.env`:** **`S3_BUCKET`**, **`S3_PREFIX`**, **`S3_SECRET_REF`** in **`env.example`** — copy into **`.env`** with your real bucket and secret reference.

**Practical checklist:**

1. **Upload traces** — Put CLEAR-compatible **`*.json`** files under a prefix (e.g. `s3://my-bucket/traces/`). Example (AWS-style CLI; COS/MinIO/GCS differ):
   ```bash
   aws s3 sync ./folder-with-json-traces/ s3://my-bucket/traces/
   ```
2. **Cluster credentials** — Eval Hub (or your platform) must be allowed to read that bucket; **`secret_ref`** in the job points at the credential object your docs describe.
3. **Job JSON** — Align **`bucket`** / **`key`** or **`path`** / **`secret_ref`** with your API; **`meta/job.json`** is a reference shape.
4. **Inside the pod** — Expect **`*.json`** under **`/test_data`** (flat) or **`/test_data/traces/`**; add **`parameters.data_dir`** only if staging differs.

More detail: **`adapters/clear/README.md`** → *Traces from S3 (deployed eval-hub)*.

## Optional: Python SDK

Below: load **`.env`**, **connect**, **list providers**, **submit**, **wait**, **save** JSON under **`examples/output/evalhub/`** (SDK only). **`curl`** from a shell does not create those files unless you redirect output yourself.


Install into **this Jupyter kernel** (fixes `No module named evalhub` and `Sentinel` / `typing_extensions` issues).

**Required:** After this cell finishes, use **Kernel → Restart Kernel** (or Restart and clear outputs). New packages are not always visible until you restart.

**Part B order:** Step 1 → **this install cell** → **restart kernel** → Step 1 → load **`.env`** → import SDK → connect → …

**If `evalhub` still missing:** In Cursor/VS Code, **Python: Select Interpreter** and choose the same **`.venv`** you use in the terminal; then restart the kernel and re-run the install cell.

In [ ]:
%pip install -q -U typing_extensions python-dotenv
import subprocess
import sys

try:
    EXAMPLES_DIR
except NameError:
    raise RuntimeError("Run Step 1 first so EXAMPLES_DIR is defined.")

_repo = EXAMPLES_DIR.parent.parent.parent.parent
_sdk = _repo / "eval-hub-sdk"
if _sdk.is_dir():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{_sdk}[adapter]"]
    )
    print("Installed eval-hub-sdk (editable from", _sdk, ")")
else:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "eval-hub-sdk[adapter]==0.1.4",
        ]
    )
    print("Installed eval-hub-sdk from PyPI")

print()
print(">>> Now: Kernel → Restart Kernel, then re-run Step 1, load .env, then Import SDK.")

Load **`.env`** after **Step 1** (needs **`EXAMPLES_DIR`**). Copy from **`env.example`** if you have not already.

In [ ]:
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError('Install python-dotenv: pip install python-dotenv')

_env_ok = load_dotenv(EXAMPLES_DIR / ".env")
print("Loaded examples/.env" if _env_ok else "No .env — copy examples/env.example to examples/.env and fill values (see Prerequisites)")

## Part B — Import SDK and read credentials

Only run this **after** the install cell above **and** a **kernel restart**.

In [ ]:
try:
    from evalhub import (
        SyncEvalHubClient,
        JobSubmissionRequest,
        BenchmarkConfig,
        ModelConfig,
        JobStatus,
    )
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        "Package 'evalhub' is not installed in THIS Jupyter kernel. "
        "1) Run the install cell above (with Step 1 done first). "
        "2) Kernel → Restart Kernel. "
        "3) Re-run Step 1, load .env, then this cell. "
        "4) Ensure the notebook kernel matches your .venv (Python: Select Interpreter). "
        "Or in a terminal: python -m pip install 'eval-hub-sdk[adapter]==0.1.4' using the same Python as the kernel."
    ) from _e

evalhub_token = os.getenv("EVALHUB_TOKEN")
if not evalhub_token:
    raise ValueError(
        "EVALHUB_TOKEN is not set. Copy examples/env.example to examples/.env and set EVALHUB_* (see Prerequisites)."
    )

evalhub_url = os.getenv("EVALHUB_URL")
if not evalhub_url:
    raise ValueError("EVALHUB_URL is not set. Set it in examples/.env (from env.example); see Prerequisites.")

evalhub_tenant = os.getenv("EVALHUB_TENANT")
if not evalhub_tenant:
    raise ValueError("EVALHUB_TENANT is not set. Set it in examples/.env (from env.example); see Prerequisites.")

## Connect to Eval Hub (health check)

In [ ]:
client = SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=evalhub_token,
    insecure=True,
    tenant=evalhub_tenant,
)
try:
    health = client.health()
    status = getattr(health, "status", None)
    if status is None and isinstance(health, dict):
        status = health.get("status")
    print(f"✓ Eval Hub health: {status}")
    extra = health.model_dump() if hasattr(health, "model_dump") else health
    if isinstance(extra, dict):
        print(f"  Version: {extra.get('version', 'unknown')}")
        print(f"  Uptime: {extra.get('uptime_seconds', 0):.1f}s")
except Exception as e:
    print(f"✗ Failed to connect: {e}")

## List providers and benchmarks

In [ ]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

### We use `ibm-clear` as `provider_id` in this notebook

List benchmarks for that provider (CLEAR exposes **`agentic-evaluation`**).

In [ ]:
provider_id = "ibm-clear"

In [ ]:
try:
    benchmarks = client.benchmarks.list(provider_id=provider_id)
    print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
    for benchmark in benchmarks:
        print(f"  - {benchmark.id}: {benchmark.name}")
except Exception as e:
    print(f"Could not list benchmarks (provider may not be registered yet): {e}")

## Submit an evaluation job

Set **`MODEL_URL`**, **`MODEL_NAME`**, **`S3_*`**, optional **`EXPERIMENT_NAME`** / **`JOB_NAME`** in **`.env`**. Submit uses **`inference_backend`: `endpoint`**. The next cells read **`os.getenv`**.

**Finding an in-cluster URL (conceptual):**

```bash
oc get svc -n <your-namespace>
# Pattern: http://<service>.<namespace>.svc.cluster.local:<port>/v1
# Use .../v1 — not .../v1/completions
```

In [ ]:
# MODEL_URL / MODEL_NAME from examples/.env (same as Part A); defaults are placeholders only.
model_url = os.getenv(
    "MODEL_URL",
    "http://YOUR_SVC.YOUR_NAMESPACE.svc.cluster.local:8000/v1",
)
model_name = os.getenv("MODEL_NAME", "example-model")
job_name = os.getenv("JOB_NAME", "clear-evalhub-job-1")
benchmark_id = "agentic-evaluation"

In [ ]:
clear_params = {
    "eval_model_name": model_name,
    "provider": "openai",
    "agent_framework": "langgraph",
    "observability_framework": "mlflow",
    "inference_backend": "endpoint",
    "agent_mode": True,
}

s3_bucket = os.getenv("S3_BUCKET", "").strip()
s3_prefix = os.getenv("S3_PREFIX", "").strip()
s3_secret = os.getenv("S3_SECRET_REF", "").strip()
if not (s3_bucket and s3_prefix and s3_secret):
    raise ValueError(
        "Set S3_BUCKET, S3_PREFIX, S3_SECRET_REF in examples/.env (same as curl test_data_ref)."
    )

test_data_ref = {
    "s3": {
        "bucket": s3_bucket,
        "key": s3_prefix,
        "secret_ref": s3_secret,
    }
}
_exp = os.getenv("EXPERIMENT_NAME", "clear-agentic-mlflow-exp").strip()
experiment = {"name": _exp} if _exp else None

job_request = JobSubmissionRequest(
    name=job_name,
    experiment=experiment,
    model=ModelConfig(url=model_url, name=model_name),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters=clear_params,
            test_data_ref=test_data_ref,
        )
    ],
)

job = client.jobs.submit(job_request)
submitted_job_id = job.id
print("✓ Job submitted successfully")
print(f"  Job ID: {submitted_job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}")

(OUTPUT_DIR_EVALHUB / "evalhub-submit-response.json").write_text(
    job.model_dump_json(indent=2), encoding="utf-8"
)
print(f"  Saved: {OUTPUT_DIR_EVALHUB / 'evalhub-submit-response.json'}")

In [ ]:
updated_job = client.jobs.get(submitted_job_id)
print(f"\n✓ Current job state: {updated_job.state}")
if updated_job.status and updated_job.status.message:
    print(f"  Message: {updated_job.status.message.message}")

## Wait for the job to complete

In [ ]:
while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(submitted_job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print("✓ Job completed successfully")

## Job results (CLEAR metrics)

Eval Hub exposes metrics under `updated_job.results.benchmarks[0].metrics` (e.g. `total_interactions`, `average_score` — see `provider.yaml`).

In [ ]:
if updated_job.results and updated_job.results.benchmarks:
    m = updated_job.results.benchmarks[0].metrics
    print("CLEAR / benchmark metrics:", m)
    (OUTPUT_DIR_EVALHUB / "evalhub-clear-metrics.json").write_text(
        json.dumps(m, indent=2, default=str), encoding="utf-8"
    )
    print(f"✓ Wrote {OUTPUT_DIR_EVALHUB / 'evalhub-clear-metrics.json'}")
else:
    print("No benchmark results on job object yet.")

In [ ]:
full_path = OUTPUT_DIR_EVALHUB / "evalhub-job-full.json"
full_path.write_text(updated_job.model_dump_json(indent=2), encoding="utf-8")
print("Wrote", full_path)

## Clean up (optional)

**`client.jobs.cancel(..., hard_delete=True)`** permanently **removes the evaluation job** from Eval Hub (not just “stop running”). Leave the line **commented** unless you really want to delete that job record.

**`client.close()`** closes the SDK’s HTTP session to Eval Hub, good hygiene when you are done with Part B.

In [ ]:
# Uncomment to permanently delete this job on Eval Hub (hard_delete=True)
# client.jobs.cancel(submitted_job_id, hard_delete=True)

## Close the client

In [ ]:
try:
    client.close()
except NameError:
    pass

## What gets saved where

Cheat sheet for **what shows up on your laptop** vs **what stays on the cluster**.

| Where | Typical artifacts |
|-------|-------------------|
| **Local `main.py`** (`EVALHUB_MODE=local`, `parameters.results_dir` + `run_name`) | **`clear_results.json`** at the run output directory (full CLEAR report). Job spec and logs are whatever your wrapper writes (e.g. this tutorial uses **`examples/output/local/`**). |
| **Notebook + Eval Hub API** (laptop) | **`evalhub-clear-metrics.json`** and related JSON under **`examples/output/evalhub/`** — metrics and job payloads returned by Eval Hub, **not** the pod’s full **`clear_results.json`**. |
| **Deployed job** (cluster) | **`clear_results.json`** is produced inside the workload; use **MLflow** (`experiment_name` / `mlflow_experiment_name`) or **OCI export** if you need that file (and **`metrics_summary.json`**) retained outside the pod. |

**This notebook:** Part A also writes **`local-job-spec.json`** and **`local-main-py.log`** under **`examples/output/local/`**. Part B may write additional snapshots (e.g. submit response, full job JSON). For MLflow / OCI on the job, see the adapter **`README.md`**.
